# Intersectoin Pipeline
Diese Pipeline liest alle eindeutigen City-Namen aus den Cityscapes-GPS-CSV-Dateien,
verarbeitet jede City nacheinander mit OSMnx (Kreuzungen/Nodes konsolidieren)
und sammelt alle Ergebnisse in einer einzigen Output-CSV.

In [2]:
import os
import gc
import osmnx as ox
import Util as util

In [3]:
# Alle eindeutigen Städte aus den bereitgestellten GPS-Dateien sammeln
cities = util.get_unique_cities_from_gps_data(
    '../output/cityscapes_coarse_gps_data.csv',
    '../output/cityscapes_fine_gps_data.csv'
)
print(f"Anzahl eindeutiger Städte aus Cityscapes: {len(cities)}")
print(cities)

Anzahl eindeutiger Städte aus Cityscapes: 50
['aachen', 'augsburg', 'bad-honnef', 'bamberg', 'bayreuth', 'berlin', 'bielefeld', 'bochum', 'bonn', 'bremen', 'cologne', 'darmstadt', 'dortmund', 'dresden', 'duisburg', 'dusseldorf', 'erfurt', 'erlangen', 'frankfurt', 'freiburg', 'hamburg', 'hanover', 'heidelberg', 'heilbronn', 'jena', 'karlsruhe', 'konigswinter', 'konstanz', 'krefeld', 'leverkusen', 'lindau', 'mainz', 'mannheim', 'monchengladbach', 'muhlheim-ruhr', 'munich', 'munster', 'nuremberg', 'oberhausen', 'saarbrucken', 'schweinfurt', 'strasbourg', 'stuttgart', 'troisdorf', 'tubingen', 'ulm', 'weimar', 'wuppertal', 'wurzburg', 'zurich']


Interessanterweise wurden konigswinter und bad-honnef gefunden

aber muhlheim-ruhr nicht also muss es in Mülheim an der Ruhr umbenannt werden damit alle stadte gefunden werden können

In [4]:
# Korrektur für Mülheim
cities[cities.index('muhlheim-ruhr')] = 'muelheim an der ruhr'

In [5]:
output_file = '../output/cityscapes_intersections.csv'

# OSMnx Konfiguration
ox.settings.use_cache = True
ox.settings.log_console = True

# Header nur einmal schreiben, danach immer anhängen
first_run = not os.path.exists(output_file)
if not first_run:
    print(f"Datei existiert bereits, neue Ergebnisse werden angehängt: {output_file}")

In [6]:
# Jede City nacheinander verarbeiten
for idx, city in enumerate(cities, start=1):
    print(f"\n[{idx}/{len(cities)}] Verarbeite City: {city}")

    # Wir versuchen zuerst die eindeutigere Abfrage "<city>, Germany",
    # wenn das fehlschlägt, nehmen wir als Fallback nur den Stadtnamen.
    region_candidates = [f"{city}, Germany", city]

    df_result = None
    used_region = None
    for candidate in region_candidates:
        df_result = util.process_region(candidate)
        if df_result is not None and not df_result.empty:
            used_region = candidate
            break

    if df_result is None or df_result.empty:
        print(f" -> Keine Daten für '{city}' erhalten (auch Fallback ohne Land schlug fehl). Überspringe.")
        # Speicher aufräumen und weiter
        if 'df_result' in locals():
            del df_result
        gc.collect()
        continue

    # 4) Export in CSV über Util-Funktion (nur wenn Daten vorhanden)
    rows_written, first_run = util.append_intersections_to_csv(
        df_result=df_result,
        city=city,
        output_file=output_file,
        used_region=used_region,
        first_run=first_run,
    )
    if rows_written > 0:
        print(f" -> Exportiert: {rows_written} Kreuzungen für {city} -> {output_file}")

    # 6) Speicher aufräumen
    if 'df_result' in locals(): del df_result
    gc.collect()

print("\nFertig! Alle eindeutigen Cityscapes-Städte wurden verarbeitet.")



[1/50] Verarbeite City: aachen
Starte Verarbeitung für: aachen, Germany
 -> 373 echte 4-Wege-Kreuzungen in aachen, Germany gefunden.
 -> Exportiert: 373 Kreuzungen für aachen -> ../output/cityscapes_intersections.csv

[2/50] Verarbeite City: augsburg
Starte Verarbeitung für: augsburg, Germany
 -> 656 echte 4-Wege-Kreuzungen in augsburg, Germany gefunden.
 -> Exportiert: 656 Kreuzungen für augsburg -> ../output/cityscapes_intersections.csv

[3/50] Verarbeite City: bad-honnef
Starte Verarbeitung für: bad-honnef, Germany
 -> 89 echte 4-Wege-Kreuzungen in bad-honnef, Germany gefunden.
 -> Exportiert: 89 Kreuzungen für bad-honnef -> ../output/cityscapes_intersections.csv

[4/50] Verarbeite City: bamberg
Starte Verarbeitung für: bamberg, Germany
 -> 178 echte 4-Wege-Kreuzungen in bamberg, Germany gefunden.
 -> Exportiert: 178 Kreuzungen für bamberg -> ../output/cityscapes_intersections.csv

[5/50] Verarbeite City: bayreuth
Starte Verarbeitung für: bayreuth, Germany
 -> 223 echte 4-Wege-Kreu

In [7]:
import folium
import pandas as pd


csv_path = '../output/cityscapes_intersections_4_strassen.csv'
df = pd.read_csv(csv_path)

desired_city = 'munich'

df_city = df[df['city'].str.lower() == desired_city.lower()]

if df_city.empty:
    print(f"Keine Daten für '{desired_city}' gefunden.")
else:
    # Mittelpunkt bestimmen
    start_lat = float(df_city.iloc[0]['latitude'])
    start_lon = float(df_city.iloc[0]['longitude'])

    m = folium.Map(location=[start_lat, start_lon], zoom_start=13)


    for row in df_city.itertuples():
        folium.CircleMarker(
            location=[row.latitude, row.longitude],
            radius=3,
            color='red',
            fill=True,
            fill_color='red',
            opacity=0.8
        ).add_to(m)

m  # Karte anzeigen